# Local Confluence RAG — ChromaDB

Purpose: connect to Confluence, read pages, clean content, chunk it, create embeddings, store them in local ChromaDB, and retrieve relevant chunks for a question.

**Install first:**
```bash
pip install requests beautifulsoup4 chromadb sentence-transformers
```

Set `CONFLUENCE_EMAIL` and `CONFLUENCE_API_TOKEN` as environment variables before running.


In [ ]:
import os
import re
import hashlib
from datetime import datetime, timezone

import requests
from bs4 import BeautifulSoup
import chromadb
from sentence_transformers import SentenceTransformer


In [ ]:
CONFLUENCE_URL = "https://YOUR-COMPANY.atlassian.net"
CONFLUENCE_EMAIL = os.getenv("CONFLUENCE_EMAIL")
CONFLUENCE_API_TOKEN = os.getenv("CONFLUENCE_API_TOKEN")
SPACE_KEY = "YOUR_SPACE_KEY"

if not CONFLUENCE_EMAIL or not CONFLUENCE_API_TOKEN:
    raise ValueError("Set CONFLUENCE_EMAIL and CONFLUENCE_API_TOKEN environment variables.")

session = requests.Session()
session.auth = (CONFLUENCE_EMAIL, CONFLUENCE_API_TOKEN)
session.headers.update({"Accept": "application/json"})
print("Confluence session created")


In [ ]:
def get_confluence_pages(space_key, limit=100):
    url = f"{CONFLUENCE_URL}/wiki/rest/api/content"
    params = {
        "spaceKey": space_key,
        "type": "page",
        "limit": limit,
        "expand": "body.storage,version,space"
    }
    response = session.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json().get("results", [])

pages = get_confluence_pages(SPACE_KEY)
print("Pages found:", len(pages))
for page in pages[:10]:
    print(page["id"], page["title"])


In [ ]:
def clean_html(html):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = soup.get_text(separator="\n", strip=True)
    return re.sub(r"\n+", "\n", text)

def chunk_text(text, chunk_size=1000, overlap=200):
    words = text.split()
    chunks = []
    start = 0
    step = max(1, chunk_size - overlap)
    while start < len(words):
        chunks.append(" ".join(words[start:start + chunk_size]))
        start += step
    return chunks


In [ ]:
def create_page_record(page):
    page_id = page["id"]
    title = page["title"]
    version = page.get("version", {}).get("number")
    html = page["body"]["storage"]["value"]
    content = clean_html(html)

    page_url = f"{CONFLUENCE_URL}/wiki/pages/{page_id}"
    content_hash = hashlib.sha256(content.encode("utf-8")).hexdigest()

    return {
        "page_id": page_id,
        "title": title,
        "version": version,
        "url": page_url,
        "content": content,
        "content_hash": content_hash,
        "last_updated": datetime.now(timezone.utc).isoformat()
    }

documents = []
for page in pages:
    record = create_page_record(page)
    if record["content"].strip():
        documents.append(record)

print("Documents prepared:", len(documents))


In [ ]:
rag_chunks = []

for doc in documents:
    for i, chunk in enumerate(chunk_text(doc["content"])):
        rag_chunks.append({
            "id": f'{doc["page_id"]}_{doc["content_hash"][:12]}_{i}',
            "text": chunk,
            "metadata": {
                "page_id": doc["page_id"],
                "title": doc["title"],
                "version": str(doc["version"] or ""),
                "url": doc["url"],
                "chunk_number": i,
                "content_hash": doc["content_hash"]
            }
        })

print("Total chunks:", len(rag_chunks))


In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [x["text"] for x in rag_chunks]
embeddings = embedding_model.encode(texts, show_progress_bar=True)

print("Embeddings created:", len(embeddings))


In [ ]:
chroma_client = chromadb.PersistentClient(path="./confluence_chroma")
collection = chroma_client.get_or_create_collection(
    name="confluence_onboarding"
)

if rag_chunks:
    collection.upsert(
        ids=[x["id"] for x in rag_chunks],
        documents=[x["text"] for x in rag_chunks],
        embeddings=[x.tolist() for x in embeddings],
        metadatas=[x["metadata"] for x in rag_chunks]
    )

print("ChromaDB chunks:", collection.count())


In [ ]:
question = "How do I request Databricks access?"

question_embedding = embedding_model.encode([question])[0]

results = collection.query(
    query_embeddings=[question_embedding.tolist()],
    n_results=5
)

for i, document in enumerate(results["documents"][0]):
    metadata = results["metadatas"][0][i]
    distance = results["distances"][0][i] if results.get("distances") else None

    print("=" * 90)
    print("TITLE:", metadata["title"])
    print("URL:", metadata["url"])
    print("DISTANCE:", distance)
    print()
    print(document)


## Next step

Add an LLM after retrieval. The LLM prompt should explicitly require answers to use only the retrieved Confluence evidence and return "I couldn't find this information in the Confluence documentation." when evidence is insufficient.
